# 2.3 — Procesamiento: índice de marginación (censal → mensual)

**Capa 1 del protocolo.** La marginación solo se observa en los censos
(2010, 2015, 2020). Esta libreta homologa los archivos municipales de
`01_ingesta/1.3-indice.-marginacion.ipynb` y revisa la interpolación mensual
que usa el panel.

## Entradas

| Ruta | Contenido |
|------|-----------|
| `data/interim/marginacion/marginacion_municipal_{2010,2015,2020}.csv` | IMM municipal de Sonora (CONAPO) |

## Salidas

- `data/interim/marginacion/marginacion_hermosillo_mensual.csv` (opcional,
  para revisión): indicadores del índice interpolados por mes.

## Checklist

- [ ] Homologar nombres de columnas entre años (`IM_2015` → `IM`, …) y verificar que los indicadores coinciden.
- [ ] Fila de Hermosillo (`26030`) presente en cada año.
- [ ] Comparabilidad: solo la serie DP2 2010-2015-2020 de CONAPO es comparable entre sí.
- [ ] Interpolación lineal 2015 → 2020; valor constante desde ene-2020.
- [ ] (Opcional) Usar 2010 si se amplía la ventana del panel (`PANEL_START_YEAR`).
- [ ] AGEB urbana (IMU 2020): pendiente de la cartografía INEGI.

## Configuración e imports

In [ ]:
from pathlib import Path
import sys

ROOT_DIR = Path.cwd().parent.parent  # notebooks/<seccion>/ -> raiz del proyecto
if str(ROOT_DIR) not in sys.path:
    sys.path.append(str(ROOT_DIR))

import pandas as pd
import matplotlib.pyplot as plt

from sonora_health_climate_eda.config import INTERIM_DATA_DIR
from sonora_health_climate_eda.panel import marginacion_hermosillo

MARG_DIR = INTERIM_DATA_DIR / "marginacion"
ANIOS_CENSO = (2010, 2015, 2020)

## Carga de los censos

In [ ]:
censos = {}
for anio in ANIOS_CENSO:
    path = MARG_DIR / f"marginacion_municipal_{anio}.csv"
    if path.exists():
        censos[anio] = pd.read_csv(path)
        print(anio, censos[anio].shape)
    else:
        print(anio, "no encontrado:", path.relative_to(ROOT_DIR))

## Homologación de columnas entre años

In [ ]:
def sin_sufijo(df: pd.DataFrame, anio: int) -> pd.DataFrame:
    """IM_2015 -> IM, GM_2015 -> GM, etc. (misma regla que panel.py)."""
    return df.rename(columns={c: c.rsplit("_", 1)[0] for c in df.columns if c.endswith(f"_{anio}")})


homologados = {anio: sin_sufijo(df, anio) for anio, df in censos.items()}
columnas = {anio: set(df.columns) for anio, df in homologados.items()}
comunes = set.intersection(*columnas.values()) if columnas else set()
print("Columnas comunes:", sorted(comunes))
for anio, cols in columnas.items():
    print(f"{anio} — solo en este año:", sorted(cols - comunes))

## Hermosillo en cada censo

In [ ]:
def fila_hermosillo(df: pd.DataFrame) -> pd.DataFrame:
    cve = "CVE_MUN" if "CVE_MUN" in df.columns else "CVE_CONCATENADA"
    return df[df[cve].astype(str).str.endswith("030")]  # misma regla que panel.py


hmo = {anio: fila_hermosillo(df) for anio, df in homologados.items()}
for anio, df in hmo.items():
    print(anio, "filas de Hermosillo:", len(df))

pd.concat(hmo, names=["anio", None]).droplevel(1).T

## Interpolación mensual

`marginacion_hermosillo()` interpola linealmente cada indicador numérico
entre 2015 y 2020 y lo deja constante después de ene-2020 (`np.clip`).

In [ ]:
marg_mensual = marginacion_hermosillo()
indicadores = [c for c in marg_mensual.columns if c != "fecha_mes"]
print("Indicadores interpolados:", indicadores)
marg_mensual.head()

In [ ]:
col = "IMN" if "IMN" in marg_mensual.columns else indicadores[0]
fig, ax = plt.subplots(figsize=(10, 3.5))
ax.plot(marg_mensual["fecha_mes"], marg_mensual[col])
ax.set_title(f"{col} mensual interpolado — Hermosillo")
plt.tight_layout()
plt.show()

# TODO: si se amplía la ventana a 2010, interpolar por tramos 2010→2015→2020
# con los tres censos (requiere ajustar panel.marginacion_hermosillo()).

## Guardar salida

In [ ]:
# TODO: guardar una vez validada la interpolación
# marg_mensual.to_csv(MARG_DIR / "marginacion_hermosillo_mensual.csv", index=False)

## Decisiones de procesamiento

Registrar cada decisión que cambie los datos y su justificación. Si la
decisión cambia el panel, implementarla en `sonora_health_climate_eda/panel.py`
(es lo que ejecuta `03_eda/3.0-integracion-espacial.ipynb`) y llevarla al
reporte final.

| Variable / periodo | Problema | Decisión | Justificación |
|--------------------|----------|----------|---------------|
| Marginación mensual | Solo se observa en censos | Interpolación lineal 2015 → 2020, constante después | Supone cambio gradual entre censos (limitación documentada) |
| `POB_TOT`, claves y nombres | No son indicadores del índice | Se excluyen de la interpolación | `POB_TOT` es un stock censal |
| _TODO_ | | | |